# edgar-extract — grammar-constrained decoding (test, BİR KEZ)

Bu defter **eğitim yapmaz**. Tek işi: çıktı şeması *zorla* dayatıldığında prompted
kolların tam-kayıt skorunun ne olduğunu ölçmek.

⛔ **Koşmadan önce `schema/CONSTRAINED_DECODING_KARARI.md` okunur.** Karar kuralı
koşudan önce yazıldı ve commit'lendi; eşiği sonradan gevşetmek geçmişte görünür.
Çubuk: fine-tuned kolun test skoru **22/36**.

Kod değişebilir `main`'den değil, **`v1.2-constrained`** etiketinden klonlanır.

In [ ]:
# 1) GPU DOĞRULAMA — tek kart, ve HANGİ kart
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

# Python < 3.14 ZORUNLU: outlines 3.14'ü desteklemiyor. Deponun kendi yerel/CI
# ortamı 3.14 olduğu için --constrained ORADA kurulamaz; burası bu yüzden var.
print("python", sys.version.split()[0])
assert sys.version_info < (3, 14), "outlines python<3.14 istiyor"

In [ ]:
# 2) KURULUM — sürümler SABİT (eğitim defteriyle AYNI yığın + outlines)
!pip install -q transformers==5.14.1 trl==1.9.2 peft==0.20.0 datasets==5.0.1 accelerate==1.14.0 bitsandbytes==0.50.0
!pip install -q outlines
# torchao ÖNCEDEN KURULU gelirse peft'in LoRA dispatcher'ı ImportError atar ve
# koşu daha kurulurken patlar. Kurulu değilse bu komut zararsız.
!pip uninstall -y -q torchao || true
import transformers, outlines, sys
print("transformers", transformers.__version__)
print("outlines OK")

In [ ]:
# 3) DEPO — değişebilir main DEĞİL, pinli etiket
import os
PIN = "v1.2-constrained"
KOK = "/content/edgar-extract"
if not os.path.exists(KOK):
    !git clone -q --branch {PIN} --depth 1 https://github.com/ozantosn24-ux/sec-filing-extraction-finetune.git {KOK}
os.chdir(KOK)
# NE koştuğunu kayda geçir: etiket oynatılamaz ama künye yine de yazılır.
!git rev-parse HEAD
!python -B src/build_sft.py | tail -3

In [ ]:
# 4) SMOKE — 135M model, 2 kayıt, TRAIN bölmesi. Test'e dokunmadan yığını sınar.
# Yerelde CPU'da geçti; buradaki soru "GPU'da ve Colab imajında da geçiyor mu".
!python -B src/predict.py --model HuggingFaceTB/SmolLM2-135M-Instruct     --split train --limit 2 --constrained -o /content/smoke_constrained.jsonl
import json, sys
sys.path.insert(0, "src")
from evaluate import parse_prediction
for ln in open("/content/smoke_constrained.jsonl", encoding="utf-8"):
    r = json.loads(ln); obj, ihlal = parse_prediction(r["raw"])
    print(r["accession"], "anahtar:", len(obj) if obj else None, "ihlal:", ihlal)
print("^ 13 anahtar ve ihlal:[] görmeden aşağı İNMEYİN")

In [ ]:
# 5) KOL A — base 1.5B (adaptörsüz) + constrained, TEST
!python -B src/predict.py --split test --constrained     -o /content/preds_base1p5b_test_constrained.jsonl

In [ ]:
# 6) KOL B — prompted 3B, 4-bit + constrained, TEST
!python -B src/predict.py --model Qwen/Qwen2.5-3B-Instruct --4bit --split test --constrained     -o /content/preds_prompted3b_test_constrained.jsonl

---
### Kontrol kolu — Drive gerektirir

Aşağıdaki hücre fine-tuned adaptörü Drive'dan okur. **Drive mount, kişisel
Drive'ın tamamına OAuth erişimi verir** (güvenlik notunun açıkça saydığı bir
güven sınırı). Kontrol kolunun işi: kısıt, zaten %100 şema-geçerli olan kola bir
şey KAYBETTİRİYOR mu? Kaybettiriyorsa prompted kolların skorları tek başına
alıntılanamaz.

Koşmazsanız karar kuralının 3. maddesi uygulanamaz; bunu README'ye
"kontrol kolu koşulmadı" diye yazın — sessizce atlamayın.

In [ ]:
# 7) KONTROL KOLU — fine-tuned + constrained (Drive mount ister)
from google.colab import drive
drive.mount("/content/drive")
ADAPTOR = "/content/drive/MyDrive/edgar-extract/lora-qwen2.5-1.5b"
import os; assert os.path.exists(ADAPTOR), f"adaptör yok: {ADAPTOR}"
!python -B src/predict.py --adapter {ADAPTOR} --split test --constrained     -o /content/preds_ft_test_constrained.jsonl

In [ ]:
# 8) SKORLA — üç kolu da, aynı harness
import glob
dosyalar = sorted(glob.glob("/content/preds_*_constrained.jsonl"))
dosyalar = [d for d in dosyalar if "smoke" not in d]
print("skorlanacak:", dosyalar)
!python -B src/evaluate.py {" ".join(dosyalar)} --split test --json-out /content/eval_constrained.json

In [ ]:
# 9) ÇIKARIM — files.download() Chrome tarafından sessizce engelleniyor (ölçüldü).
# Çalışan yol: tek satır halinde yazdır, çıktıyı kopyala.
# ÖNCE yukarıdaki hücrelerin çıktısını temizleyin, yoksa 50k sınırına takılır.
import glob, json
for yol in sorted(glob.glob("/content/preds_*_constrained.jsonl")) + ["/content/eval_constrained.json"]:
    if "smoke" in yol:
        continue
    icerik = open(yol, encoding="utf-8").read()
    tek_satir = icerik.replace("
", "|LN|")
    print(f"===DOSYA=== {yol} ===UZUNLUK=== {len(icerik)}")
    print(tek_satir)
    print("===SON===")